In [1]:
import random
from cgra import *
from kernels import *

In [2]:
# Global variables
CGRA_N_ROWS = 3
CGRA_N_COLS = 3
SIZE = 60

# Adress
first_addr = 20000

# Benchmark
kernel_name = f"benchmarks/compigra_blas_paper/blas/Kalman_2/{CGRA_N_ROWS}x{CGRA_N_COLS}/"
version = f"_out_{CGRA_N_COLS}_I{SIZE}_J{SIZE}_K{SIZE}"

In [3]:
# ------------------------------------
#           FUNCTIONS
# ------------------------------------
def configMemory(A, Q, AP, NI):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------            
    first_addr_A = first_addr
    first_addr_Q = first_addr_A + NI*NI*4
    first_addr_AP = first_addr_Q + NI*NI*4
    first_addr_AT = first_addr_AP + NI*NI*4
    first_addr_APA = first_addr_AT + NI*NI*4
    first_addr_P = first_addr_APA + NI*NI*4

    config_vals = [[] for i in range(CGRA_N_COLS)]

    # void Kalman_2(int A[NI][NI], int Q[NI][NI], int AP[NI][NI], int AT[NI][NI], int APA[NI][NI], int P[NI][NI])
    # 0 : first_addr_A
    # 1 : first_addr_Q
    # 2 : first_addr_AP
    # 3 : first_addr_AT
    # 4 : first_addr_APA
    # 5 : first_addr_P

    if version == "_out_3_I24_J24_K24":
        config_vals[0] = [first_addr_P, first_addr_AT, first_addr_AP] # 5, 3, 2
        config_vals[1] = [first_addr_Q, first_addr_A, first_addr_AT, first_addr_APA] # 1, 0, 3, 4
        config_vals[2] = [first_addr_APA] # 4
    if version == "_out_3_I60_J60_K60":
        config_vals[0] = [first_addr_P, first_addr_AT, first_addr_AP] # 5, 3, 2
        config_vals[1] = [first_addr_APA, first_addr_A, first_addr_AT, first_addr_APA] # 4, 0, 3, 4
        config_vals[2] = [first_addr_Q] # 1       
    

    addr_config_loads = [0 for i in range(CGRA_N_COLS)]
    for i in range(CGRA_N_COLS):
        kernel_add_memory_region(kernel_name, addr_config_loads[i], config_vals[i], version=version)
        if i < CGRA_N_COLS -1:
            addr_config_loads[i+1] = addr_config_loads[i] + len(config_vals[i])*4
            
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A, version=version)
    kernel_add_memory_region(kernel_name, first_addr_Q, Q, version=version)
    kernel_add_memory_region(kernel_name, first_addr_AP, AP, version=version)

    # Config data address for direct loads
    return addr_config_loads

def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

def runKernel(load_addrs, max_it=1000, pr=["ROUT","INST"], printVal=1):
    # Run kernel
    run(kernel_name, pr=pr, load_addrs=load_addrs, version=version, limit=max_it, printVal=printVal)

def getResult(first_addr, length):
    result = [0 for _ in range(length)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr) and (int(row[0]) < first_addr + length*4):
                    result[int((int(row[0]) - first_addr)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [4]:
# --------------------------------------------
#               DATA
# --------------------------------------------
data = np.load(kernel_name + f"data/data_{SIZE}.npz")

A = data["A"]
Q = data["Q"]
AP = data["AP"]
NI = int(data["NI"])

# Expected results
AT_expected = data["AT_expected"]
APA_expected = data["APA_expected"]
P_expected = data["P_expected"]

print(f"Testing Karlman_2 sizes : {NI}")

Testing Karlman_2 sizes : 60


In [5]:
load_addrs = configMemory(A, Q, AP, NI)

In [6]:
runKernel(load_addrs, max_it=200000000, printVal=0)
#estimatedConfigCycles(kernel_name, version)

EXECUTION LIMIT REACHED ( 2000000 steps)
Extend the execution by calling the run with argument limit=<steps>.
END


In [7]:
# Get result from CGRA
first_addr_AT_res = first_addr + NI*NI*4*3
AT_result = getResult(first_addr_AT_res, NI*NI)

first_addr_APA_res = first_addr + NI*NI*4*4
APA_result = getResult(first_addr_APA_res, NI*NI)

first_addr_P_res = first_addr + NI*NI*4*5
P_result = getResult(first_addr_P_res, NI*NI)

# Check result correctness
print("Check AT result:")
errors = 0
err_idx = []
for i in range(len(AT_expected)):
    if AT_expected[i] != AT_result[i]:
        errors += 1
        err_idx.append(i)
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(AT_expected, 1, NI)
    print("CGRA: ")
    printAsMatrix(AT_result, 1, NI)
    print("Errors are: Exp : CGRA")
    for i in err_idx:
        print(f"Idx[{i}] {AT_expected[i]} : {AT_result[i]}")
else:
    print("OK")

print("Check APA result:")
errors = 0
err_idx = []
for i in range(len(APA_expected)):
    if APA_expected[i] != APA_result[i]:
        errors += 1
        err_idx.append(i)
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(APA_expected, NI, NI)
    print("CGRA: ")
    printAsMatrix(APA_result, NI, NI)
    print("Errors are: Exp : CGRA")
    for i in err_idx:
        row = int(i/NI)
        col = i%NI
        print(f"Idx[{row}][{col}] {APA_expected[i]} : {APA_result[i]}")
else:
    print("OK")

print("Check P result:")
errors = 0
err_idx = []
for i in range(len(P_expected)):
    if P_expected[i] != P_result[i]:
        errors += 1
        err_idx.append(i)
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(P_expected, NI, NI)
    print("CGRA: ")
    printAsMatrix(P_result, NI, NI)
    print("Errors are: Exp : CGRA")
    for i in err_idx:
        row = int(i/NI)
        col = i%NI
        print(f"Idx[{row}][{col}] {P_expected[i]} : {P_result[i]}")
else:
    print("OK")



Check AT result:
Err: 6
Expected: 
[ 5  2 -1  1  5  5  4  5  1 -4 -1  3  2 -1  0 -3 -4 -1 -3  0 -2  3  1  1
 -2  1  2 -4 -1 -1 -3  5  5  3  0 -5  3 -5 -5  5  4  3  2 -1  4 -3  0  2
  4  0  4  3 -5 -5 -1  0 -3 -2 -5  5]
CGRA: 
[5, 2, -1, 1, 5, 5, 4, 5, 1, -4, -1, 3, 2, -1, 0, -3, -4, -1, -3, 0, -2, 3, 1, 1, -2, 1, 2, -4, -1, -1, -3, 5, 5, 3, 0, -5, 3, -5, -5, 5, 4, 3, 2, -1, 4, -3, 0, 2, 4, 0, 4, 3, -5, -5, -1, 0, -3, -2, -5, 5]
Errors are: Exp : CGRA
Idx[456] 0 : 48800
Idx[457] -5 : 63200
Idx[458] 4 : 77600
Idx[462] 4 : 77600
Idx[463] -2 : 34400
Idx[464] 1 : 92000
Check APA result:
Err: 3579
Expected: 
[  56  176    5   96   46  200  -36   57   37  -26   11  -10  -62    0
   47   82  -13  -66  112    4   42  -65  -17  -60  -61  -26  -65 -146
 -161   35   21   41  -16   56  -18  192  -72  -11   36 -138   43 -151
  -46   94    0  -85  -27   32 -173  -61  -53 -104  -24  -12   66    2
  106   18   37  132]
[ -59  -42   -8   81 -125   34  -19 -136   50  -52  -80  -46    2  -12
   -8  -36  -